# NLG Sampling Strategies

## Greedy search, Beam search, Top-K sampling and Top-p sampling.

GPT-2 is an autoregressive, decoder-only Transformer trained with a causal language modeling (CLM) objective: at each step it outputs a probability distribution over the next token given all previous tokens. This notebook uses it to compare decoding strategies.

- Greedy search always picks the highest-probability next token
- Beam search keeps several likely sequences to improve overall fluency.
- Top-k sampling limits choices to the k most likely tokens
- Top-p sampling selects from the smallest set of tokens whose cumulative probability passes a threshold.

Temperature rescales the logits before the softmax ($z_i / T$): lower values sharpen the distribution, higher values flatten it.

Generation stops when any of these happens: the sequence reaches the maximum length, the end-of-sequence token (`<eos>`) is produced, or a time limit is hit.

https://huggingface.co/blog/how-to-generate

In [1]:
!#pip install --upgrade transformers torch

In [2]:
import torch as tf
from transformers import GPT2LMHeadModel, GPT2Tokenizer

In [3]:
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token
model = GPT2LMHeadModel.from_pretrained("gpt2")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [4]:
# Prepare inputs
inputs = tokenizer(
    "The world's best cuisine is",
    return_tensors="pt",
    padding=True
)

input_ids=inputs["input_ids"]

In [5]:
# Greedy output
greedy_output = model.generate(
    input_ids=input_ids,
    max_length=50,
    pad_token_id=tokenizer.eos_token_id, # silence pad warnings
    attention_mask=inputs["attention_mask"]    
)

print(tokenizer.decode(greedy_output[0], skip_special_tokens=True))

The world's best cuisine is made up of a wide range of ingredients, from fresh vegetables to fresh fruits and vegetables.

The world's best cuisine is made up of a wide range of ingredients, from fresh vegetables to fresh fruits and vegetables.


In [6]:
# Activate beam search and early_stopping
beam_output = model.generate(
    input_ids, 
    max_length=50, 
    num_beams=5, 
    early_stopping=True,
    pad_token_id=tokenizer.eos_token_id,  # silence pad warnings
    attention_mask=inputs["attention_mask"]    
)

print("Output:\n" + 100 * '-')
print(tokenizer.decode(beam_output[0], skip_special_tokens=True))

Output:
----------------------------------------------------------------------------------------------------
The world's best cuisine is made up of the best ingredients.

The world's best cuisine is made up of the best ingredients.

The world's best cuisine is made up of the best ingredients.

The world's best cuisine


In [7]:
# set no_repeat_ngram_size to 2
beam_output = model.generate(
    input_ids, 
    max_length=50, 
    num_beams=5, 
    no_repeat_ngram_size=2, 
    early_stopping=True,
    pad_token_id=tokenizer.eos_token_id,  # silence pad warnings   
    attention_mask=inputs["attention_mask"]    
)

print("Output:\n" + 100 * '-')
print(tokenizer.decode(beam_output[0], skip_special_tokens=True))

Output:
----------------------------------------------------------------------------------------------------
The world's best cuisine is made up of the best ingredients, and it's not just about the food. It's about how you eat it.

It's also about what you do with your time. If you don't have time to


In [8]:
# set return_num_sequences > 1
beam_outputs = model.generate(
    input_ids, 
    max_length=50, 
    num_beams=5, 
    no_repeat_ngram_size=2, 
    num_return_sequences=5, 
    attention_mask=inputs["attention_mask"],     
    early_stopping=True,
    pad_token_id=tokenizer.eos_token_id  # silence pad warnings           
)

# now we have 3 output sequences
print("Output:\n" + 100 * '-')
for i, beam_output in enumerate(beam_outputs):
  print("{}: {}".format(i, tokenizer.decode(beam_output, skip_special_tokens=True)))

Output:
----------------------------------------------------------------------------------------------------
0: The world's best cuisine is made up of the best ingredients, and it's not just about the food. It's about how you eat it.

It's also about what you do with your time. If you don't have time to
1: The world's best cuisine is made up of the best ingredients, and it's not just about the food. It's about how you eat it.

It's also about what you do with your time. If you don't have time,
2: The world's best cuisine is made up of the best ingredients, and it's not just about the food. It's about how you eat it.

It's also about what you do with your time. If you don't have the time
3: The world's best cuisine is made up of the best ingredients, and it's not just about the food. It's about how you eat it.

It's also about what you do with your time. If you don't have time for
4: The world's best cuisine is made up of the best ingredients, and it's not just about the food. It's abou

In [9]:
# set seed to reproduce results
tf.random.set_seed(0)

# activate sampling and deactivate top_k by setting top_k sampling to 0
sample_output = model.generate(
    input_ids, 
    do_sample=True, 
    max_length=50, 
    top_k=0,
    pad_token_id=tokenizer.eos_token_id,  # silence pad warnings        
    attention_mask=inputs["attention_mask"]
)
print("Output:\n" + 100 * '-')
print(tokenizer.decode(sample_output[0], skip_special_tokens=True))

Output:
----------------------------------------------------------------------------------------------------
The world's best cuisine is produced at the top of the world's richest restaurants, boasting ample opportunities to enjoy all of your favorite kinds of healthy and tasty food en route to other markets. Our sustainable street food menu at Spotlight Supermarkets and Cook Ser


In [10]:
# set seed to reproduce results
tf.random.set_seed(0)

# use temperature to decrease the sensitivity to low probability candidates
sample_output = model.generate(
    input_ids, 
    do_sample=True, 
    max_length=50, 
    top_k=0, 
    temperature=0.7,
    pad_token_id=tokenizer.eos_token_id,  # silence pad warnings  
    attention_mask=inputs["attention_mask"]
)

print("Output:\n" + 100 * '-')
print(tokenizer.decode(sample_output[0], skip_special_tokens=True))

Output:
----------------------------------------------------------------------------------------------------
The world's best cuisine is from around the world, and if we ask you how to get there, you'll get some tips.

1. The best way to get to this destination is in more than one way:

A.


In [11]:
# set seed to reproduce results. Feel free to change the seed though to get different results
tf.random.set_seed(0)

# set top_k to 50
sample_output = model.generate(
    input_ids, 
    do_sample=True, 
    max_length=50, 
    top_k=50,
    pad_token_id=tokenizer.eos_token_id, # silence pad warnings  
    attention_mask=inputs["attention_mask"]    
)

print("Output:\n" + 100 * '-')
print(tokenizer.decode(sample_output[0], skip_special_tokens=True))

Output:
----------------------------------------------------------------------------------------------------
The world's best cuisine is always packed at the very right time to bring you the next great meal, no matter how good the previous one sounded. What started out as a small business in the mid 1990s has grown into one of the country's


In [12]:
# set seed to reproduce results.
tf.random.set_seed(0)

# deactivate top_k sampling and sample only from 92% most likely words
sample_output = model.generate(
    input_ids, 
    do_sample=True, 
    max_length=50, 
    top_p=0.92, 
    top_k=0,
    pad_token_id=tokenizer.eos_token_id,  # silence pad warnings
    attention_mask=inputs["attention_mask"]    
)

print("Output:\n" + 100 * '-')
print(tokenizer.decode(sample_output[0], skip_special_tokens=True))

Output:
----------------------------------------------------------------------------------------------------
The world's best cuisine is distilled from Cheolink's most iconic cooking style – lovingly executed on-the-plate, and designed to open your mouth to meals that taste the same way your mother would.

Alter your breakfast menus


In [13]:
# set seed to reproduce results.
tf.random.set_seed(0)

# set top_k = 50 and set top_p = 0.95 and num_return_sequences = 3
sample_outputs = model.generate(
    input_ids,
    do_sample=True, 
    max_length=50, 
    top_k=50, 
    top_p=0.95, 
    num_return_sequences=3,
    pad_token_id=tokenizer.eos_token_id  # silence pad warnings            
)

print("Output:\n" + 100 * '-')
for i, sample_output in enumerate(sample_outputs):
  print("{}: {}".format(i, tokenizer.decode(sample_output, skip_special_tokens=True)))

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Output:
----------------------------------------------------------------------------------------------------
0: The world's best cuisine is now available by serving up fresh, low-key, delicious dishes to those who are out of shape.

I've made an extensive list of our most delicious, original and most beautiful desserts in honor of the holiday
1: The world's best cuisine is now free and open to the public. The World Famous Baked Chicken is one of the most famous and celebrated British food in London. Baked chicken is a British favourite and always a favourite among travellers at The Beech
2: The world's best cuisine is not really your cup of tea—it's a soup made with fish, eggs, carrots, and onions. We think fish tastes better than soup—but it's also quite healthy and delicious. The fact that we live


In [14]:
# set seed to reproduce results.
tf.random.set_seed(0)

# Setting multiple parameters
sample_outputs = model.generate(
    input_ids, 
    max_length=50, 
    num_beams=5,            # Number of beams for beam search
    num_return_sequences=5, # Number of sequences to return
    no_repeat_ngram_size=2, # Prevent repetition of n-grams
    early_stopping=True,    # Stop early if the model reaches an end token
    pad_token_id=tokenizer.eos_token_id,  # Silence pad warnings
    do_sample=True,         # Enable sampling for diversity
    temperature=0.7,        # Controls randomness (lower = more deterministic)
    top_p=0.9,              # Nucleus sampling (top-p sampling)
    top_k=50,                # Top-k sampling (limits the number of possible next tokens)
    attention_mask=inputs["attention_mask"],     
)

print("Output:\n" + 100 * '-')
for i, sample_output in enumerate(sample_outputs):
  print("{}: {}".format(i, tokenizer.decode(sample_output, skip_special_tokens=True)))

Output:
----------------------------------------------------------------------------------------------------
0: The world's best cuisine is not the only thing you can do to improve your health, but it is one of the most important things to do.

If you have any questions, please feel free to reach out to me on Twitter, Facebook
1: The world's best cuisine is not the only thing you can do to improve your health, but it is one of the most important things to do.

If you have any questions, please feel free to reach out to me on Twitter or email
2: The world's best cuisine is not the only thing you can do to improve your health, but it is one of the most important things to do.

If you have any questions, please feel free to reach out to us.
3: The world's best cuisine is not the only thing you can do to improve your health, but it is one of the most important things to do.

If you have any questions, please feel free to contact us.
4: The world's best cuisine is not the only thing you can

## Summary & Key Takeaways

1. **Decoding Algorithms**: Greedy search is fast but deterministic; Beam Search optimizes sequence probability; Top-K and Top-P (Nucleus) filter low-probability tails.
2. **Temperature Control**: Scales logit entropy ($z_i / T$) to govern exploration vs exploitation.
3. **Repetition Penalties**: Penalize previously generated tokens to prevent degeneration loops in long-form generation.
4. **KV Caching**: Caching Key and Value projections from previous steps avoids recomputing attention over the whole prefix at every step (see notebook 8.7, GPU inference optimization).
5. **Autoregressive Generation**: Decoder-only Transformers predict the next-token distribution conditioned on all preceding tokens; every strategy above only changes how a token is chosen from that distribution.
